# TrackX AI - features and classifier (SIMULATED data)

> **All waveforms in this notebook are simulated.** Numbers here only show the software pipeline works.
> They are **not** real-world performance. Real results need real labelled data and validation against established inspection methods.

Run from the `notebooks/` folder after `python -m src.simulate` (or run the first cell).

In [ ]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pathlib import Path
from src.simulate import generate_dataset, make_waveform, FS
from src.features import feature_matrix, FEATURE_NAMES
from src.classify import TrackXClassifier, evaluate, INCONCLUSIVE
from sklearn.model_selection import GroupShuffleSplit

## 1. Simulated waveforms

In [ ]:
rng = np.random.default_rng(1)
fig, axs = plt.subplots(1, 3, figsize=(14, 3), sharey=True)
for ax, k in zip(axs, ['normal', 'defect', 'bad_coupling']):
    w, _ = make_waveform(k, rng)
    ax.plot(np.arange(len(w))/FS*1e6, w, lw=.7); ax.set_title(k + ' (SIMULATED)'); ax.set_xlabel('time (us)')
plt.tight_layout(); plt.show()

## 2. Dataset and feature extraction

In [ ]:
X_wave, meta = generate_dataset(n_per_class=300, n_bad=60, seed=42)
X = feature_matrix(X_wave, FS)
X['kind'] = meta['kind'].values
X.groupby('kind')[['peak_amp', 'attenuation_ratio', 'n_echoes', 'snr_db']].mean().round(3)

## 3. Train Random Forest (held-out *sessions*, not random rows)

In [ ]:
clean = (meta['kind'] != 'bad_coupling').values
Xc = X[clean].reset_index(drop=True); mc = meta[clean].reset_index(drop=True)
tr, te = next(GroupShuffleSplit(1, test_size=.3, random_state=0).split(Xc, mc['has_defect'], groups=mc['session']))
clf = TrackXClassifier().fit(Xc.iloc[tr], mc['has_defect'].iloc[tr])

imp = pd.Series(clf.model.feature_importances_, index=FEATURE_NAMES).sort_values()
imp.plot.barh(title='Feature importance (SIMULATED data)'); plt.show()

## 4. Evaluate with the Inconclusive rule
Test set = unseen sessions + all bad-coupling captures (which should become *Inconclusive*).

In [ ]:
X_test = pd.concat([Xc.iloc[te], X[~clean]], ignore_index=True)
y_test = np.concatenate([mc['has_defect'].iloc[te].values, np.zeros((~clean).sum(), dtype=int)])
preds = clf.predict_df(X_test)
print(evaluate(y_test, preds))
print(pd.Series(preds).value_counts())

## 5. Caveats
* The simulator is a toy model, so the classes are easy to separate - high scores here are **expected and meaningless** for real rails.
* Real data will include joints, welds, temperature, vibration and coupling effects that need location-wise data and compensation.
* Next step: replace `data/simulated/` with labelled measurements from an NDT lab and re-run with session-wise splits.